# Xarray-Spatial Streams: D-inf and MFD stream ordering and link segmentation

Stream network analysis extracts channel topology from a DEM using flow routing models. xarray-spatial supports three routing variants: D8 (single steepest neighbor), D-infinity (continuous angle between two neighbors, Tarboton 1997), and MFD (flow fractions to all downslope neighbors). This notebook compares stream ordering and link segmentation across all three.

### What you'll build

1. Generate synthetic terrain and compute flow directions for D8, D-inf, and MFD
2. Compare Strahler stream ordering across all three routing models
3. Compare Shreve magnitude across routing models
4. Segment stream links and count network topology differences

![Stream analysis preview](images/stream_analysis_preview.png)

**Jump to a section:**
[Flow directions](#Flow-directions) | [Strahler ordering](#Strahler-ordering) | [Shreve magnitude](#Shreve-magnitude) | [Stream links](#Stream-links)

Standard imports plus flow direction, accumulation, stream ordering, and link functions for all three routing models.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, LogNorm
from matplotlib.patches import Patch

from xrspatial import generate_terrain, fill, erode, hillshade
from xrspatial import flow_direction, flow_direction_dinf, flow_direction_mfd
from xrspatial import flow_accumulation, flow_accumulation_mfd
from xrspatial import stream_order, stream_link
from xrspatial import stream_order_dinf, stream_link_dinf
from xrspatial import stream_order_mfd, stream_link_mfd

## Synthetic terrain

A 400x400 ridged-noise DEM with domain warping, heavily eroded with 500,000 hydraulic erosion iterations to carve realistic drainage channels. After erosion we fill any remaining depressions and add sub-millimeter noise to break flats, so every cell has a continuous downhill path to the grid edge.

In [ ]:
W, H = 400, 400
template = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'],
                        coords={'y': np.linspace(0, 1000, H),
                                'x': np.linspace(0, 1000, W)})
terrain = generate_terrain(template, x_range=(0, 1000), y_range=(0, 1000),
                           seed=42, zfactor=6000,
                           noise_mode='ridged', warp_strength=0.4)

# Heavy hydraulic erosion to carve drainage channels
terrain = erode(terrain, iterations=500_000, seed=42)

# Fill remaining depressions and break flats with sub-mm noise
dem = fill(terrain)
rng = np.random.RandomState(42)
dem.values += rng.uniform(0, 0.001, dem.shape)
dem = fill(dem)
rng2 = np.random.RandomState(123)
dem.values += rng2.uniform(0, 1e-6, dem.shape)

# Hillshade for basemap overlays
hs = hillshade(dem)

def plot_basemap(ax):
    """Plot faint hillshade basemap."""
    hs.plot.imshow(ax=ax, cmap='gray', add_colorbar=False, alpha=0.4)

print(f"Elevation range: {float(np.nanmin(dem.values)):.0f} to "
      f"{float(np.nanmax(dem.values)):.0f}")

fig, ax = plt.subplots(figsize=(10, 7.5))
dem.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                cbar_kwargs={'label': 'Elevation', 'shrink': 0.7})
ax.set_title('Synthetic elevation (ridged noise, 500k erosion iterations)')
ax.set_axis_off()
plt.tight_layout()

## Flow directions

Compute all three routing models from the filled elevation surface. D8 picks one neighbor, D-inf distributes flow between two, and MFD distributes to all downslope neighbors. The log-scale accumulation maps below show how each model concentrates (or disperses) drainage.

In [ ]:
# D8
fd_d8 = flow_direction(dem)
fa_d8 = flow_accumulation(fd_d8)

# D-infinity
fd_dinf = flow_direction_dinf(dem)

# MFD
fd_mfd = flow_direction_mfd(dem)
fa_mfd = flow_accumulation_mfd(fd_mfd)

print(f'D8 flow dir shape:   {fd_d8.shape}')
print(f'D-inf angles shape:  {fd_dinf.shape}')
print(f'MFD fractions shape: {fd_mfd.shape}')
print(f'D8 accumulation:     1 to {np.nanmax(fa_d8.values):.0f}')
print(f'MFD accumulation:    1 to {np.nanmax(fa_mfd.values):.0f}')

water_cmap = LinearSegmentedColormap.from_list('water', ['#6baed6', '#08306b'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes, [fa_d8, fa_mfd], ['D8', 'MFD']):
    plot_basemap(ax)
    data.plot.imshow(ax=ax, cmap=water_cmap, alpha=0.85,
                     norm=LogNorm(vmin=1, vmax=float(np.nanmax(data.values))),
                     add_colorbar=True,
                     cbar_kwargs={'label': 'Upstream cells (log)', 'shrink': 0.7})
    ax.set_title(f'{title} flow accumulation')
    ax.set_axis_off()
plt.tight_layout()

## Strahler ordering

Strahler stream order assigns order 1 to headwater channels, and increments when two channels of equal order join. The three routing models produce different junction sets, so their Strahler numbers diverge: D8 gives the sharpest single-path channels, D-inf creates more split points, and MFD creates the most because it routes flow to all downslope neighbors. Each panel below uses its own color scale since the order ranges differ substantially.

In [ ]:
threshold = 200

# D8 stream order
so_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='strahler')

# D-inf stream order
so_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='strahler')

# MFD stream order
so_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='strahler')

for label, data in [('D8', so_d8), ('D-inf', so_dinf), ('MFD', so_mfd)]:
    n = int(np.sum(~np.isnan(data.values)))
    mx = int(np.nanmax(data.values)) if n > 0 else 0
    print(f'{label:6s}: {n:5d} stream cells, max Strahler order {mx}')

In [ ]:
stream_cmap = LinearSegmentedColormap.from_list('stream', ['#6baed6', '#08306b'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, data, title in zip(axes, [so_d8, so_dinf, so_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    plot_basemap(ax)
    vmax = float(np.nanmax(data.values))
    display = xr.where(data.isnull(), np.nan, data)
    display.plot.imshow(ax=ax, cmap=stream_cmap, vmin=1, vmax=vmax,
                        alpha=0.9, add_colorbar=True,
                        cbar_kwargs={'label': 'Strahler order', 'shrink': 0.6})
    ax.set_title(f'Strahler order ({title}), max={int(vmax)}')
    ax.set_axis_off()

plt.tight_layout()

# Save preview
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/stream_analysis_preview.png', bbox_inches='tight', dpi=120)

<div class="alert alert-block alert-info">
<b>MFD inflates Strahler order.</b> Because MFD splits flow to all downslope neighbors, the stream network has far more junction points than D8 or D-inf. Each junction is a potential order increment, so MFD Strahler numbers can be 10 to 50 times higher than D8 for the same DEM. D8 Strahler order is the standard metric for comparing stream hierarchy across studies.
</div>

## Shreve magnitude

Shreve magnitude sums the count of upstream headwater channels at each point. Higher values indicate more upstream contributing area. The log scale makes the full range visible.

In [ ]:
sv_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='shreve')
sv_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='shreve')
sv_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='shreve')

for label, data in [('D8', sv_d8), ('D-inf', sv_dinf), ('MFD', sv_mfd)]:
    n = int(np.sum(~np.isnan(data.values)))
    mx = int(np.nanmax(data.values)) if n > 0 else 0
    print(f'{label:6s}: {n:5d} stream cells, max Shreve magnitude {mx}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, data, title in zip(axes, [sv_d8, sv_dinf, sv_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    plot_basemap(ax)
    vmax = float(np.nanmax(data.values))
    display = xr.where(data.isnull(), np.nan, data)
    log_display = xr.DataArray(np.log1p(display.values),
                               dims=data.dims, coords=data.coords)
    log_display.plot.imshow(ax=ax, cmap='viridis', alpha=0.9,
                            add_colorbar=True,
                            cbar_kwargs={'label': 'log(1 + magnitude)',
                                         'shrink': 0.6})
    ax.set_title(f'Shreve magnitude ({title}), max={int(vmax)}')
    ax.set_axis_off()

plt.tight_layout()

## Stream links

Link segmentation assigns a unique ID to each channel segment between junctions. Because MFD distributes flow to all downslope neighbors, almost every stream cell becomes a junction (in-degree >= 2), fragmenting the network into thousands of single-cell links. D8 and D-inf produce far fewer, longer segments.

In [ ]:
sl_d8 = stream_link(fd_d8, fa_d8, threshold=threshold)
sl_dinf = stream_link_dinf(fd_dinf, fa_d8, threshold=threshold)
sl_mfd = stream_link_mfd(fd_mfd, fa_mfd, threshold=threshold)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, data, title in zip(axes, [sl_d8, sl_dinf, sl_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    plot_basemap(ax)
    # Multiply by a prime before mod to break spatial color banding
    # (position-based IDs create vertical stripes with plain mod 20)
    display = xr.where(data.isnull(), np.nan, np.mod(data * 7919, 20) + 1)
    display.plot.imshow(ax=ax, cmap='tab20', alpha=0.9, add_colorbar=False)
    ax.set_title(f'Stream links ({title})')
    ax.set_axis_off()

plt.tight_layout()

for label, data in [('D8', sl_d8), ('D-inf', sl_dinf), ('MFD', sl_mfd)]:
    n_links = len(np.unique(data.values[~np.isnan(data.values)]))
    n_stream = int(np.sum(~np.isnan(data.values)))
    avg_len = n_stream / max(n_links, 1)
    print(f'{label:6s}: {n_links:4d} links, {n_stream:6d} stream cells, '
          f'{avg_len:.1f} cells/link avg')

<div class="alert alert-block alert-info">
<b>Choosing a routing model.</b> D8 is fastest and produces the sharpest single-path channels. D-inf gives smoother flow angles and is better for slope-area analysis. MFD distributes flow most realistically on flat terrain but produces wider, more diffuse channel networks. For stream ordering, D8 is usually sufficient. For detailed hillslope hydrology, MFD or D-inf are preferable.
</div>

### References

- Tarboton, D. G. (1997). [A new method for the determination of flow directions and upslope areas in grid digital elevation models](https://doi.org/10.1029/96WR03137). *Water Resources Research*, 33(2), 309-319.
- [Strahler stream order (Wikipedia)](https://en.wikipedia.org/wiki/Strahler_number)
- [xrspatial.stream_order API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.stream_order.html)